# M1.7 · Proyecto: Chef personal 👩‍🍳

**Lección oficial:** *Personal chef (proyecto del módulo)* · **Notebook original:** `module-1/1.5_personal_chef.ipynb`

## 🎯 Qué vas a construir
Un **chef personal** 👩‍🍳: le dices qué ingredientes tienes en casa, busca recetas en internet y recuerda la conversación.

Junta **todo** lo del módulo:

| Pieza | Lección | Para qué |
|---|---|---|
| Modelo (`get_model`) | M1.1 | El cerebro |
| System prompt | M1.2 | El guion: "eres un chef" |
| Tool `web_search` | M1.3 y M1.4 | Buscar recetas reales |
| Checkpointer + thread | M1.5 | Recordar los ingredientes |

🧠 **Mnemotecnia M-P-T-M:** todo agente se arma con **M**odelo, **P**rompt, **T**ools y **M**emoria.

## 🛠️ Paso 0 · Preparar el notebook

Esta celda hace tres cosas por ti:
1. Encuentra la carpeta principal del repo (donde vive `local_model.py`).
2. Lee tu archivo `.env` (tus llaves y la dirección del servidor) **sin mostrarlas**.
3. Revisa el **semáforo** del servidor Ollama, que es **compartido**:
   - 🟢 libre → adelante.
   - 🟡 hay un modelo cargado → si fuiste tú hace un rato, sigue; si no, espera.
   - 🔴 sin conexión → revisa `OLLAMA_BASE_URL` en tu `.env`.

> Corre las celdas **en orden, de arriba hacia abajo** (Shift + Enter). El número `[n]` a la izquierda te dice en qué orden se ejecutaron.

In [ ]:
# 🟢 ADAPTADO LOCAL: preparar entorno (ver ayuda.py)
from ayuda import preparar, ver, semaforo
preparar()

from local_model import get_model   # se importa DESPUÉS de cargar el .env

## 1 · La tool (T)
La misma búsqueda web de la lección M1.4.

In [ ]:
from typing import Any, Dict
from langchain.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for recipes and current information."""
    return tavily_client.search(query, max_results=3)

## 2 · El prompt (P)
El prompt original está en inglés. Nuestra versión es en español y con **pasos numerados**: a los modelos pequeños les ayuda mucho que las reglas sean concretas.

In [ ]:
# 🔸 ORIGINAL DEL CURSO (en inglés, más general)
# system_prompt = """
# You are a personal chef. The user will give you a list of ingredients they have left over in their house.
# Using the web search tool, search the web for recipes that can be made with the ingredients they have.
# Return recipe suggestions and eventually the recipe instructions to the user, if requested.
# """

# 🟢 ADAPTADO LOCAL: en español y con pasos claros
system_prompt = """
Eres un chef personal. El usuario te dará los ingredientes que tiene en casa.
1. Usa web_search para buscar recetas con esos ingredientes.
2. Propón máximo 3 recetas: nombre, una línea de descripción y la URL de la fuente.
3. Si una receta necesita ingredientes que el usuario no tiene, dilo.
4. Da las instrucciones completas solo si el usuario las pide.
Responde en español y de forma breve.
"""

## 3 · Armar el agente (M + P + T + M)

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre
# agent = create_agent(model="gpt-5-nano", tools=[web_search], system_prompt=system_prompt, checkpointer=InMemorySaver())

# 🟢 ADAPTADO LOCAL
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=get_model("gemma4:latest"),   # M: modelo
    system_prompt=system_prompt,        # P: prompt
    tools=[web_search],                 # T: tools
    checkpointer=InMemorySaver(),       # M: memoria
)
config = {"configurable": {"thread_id": "cocina-1"}}

## 4 · ¡A cocinar! 🍳
⏳ Buscar + leer resultados + responder en CPU puede tardar 1 a 3 minutos.

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Tengo pollo, arroz, cebolla y zanahoria. ¿Qué puedo cocinar?")]},
    config,
)
print(response["messages"][-1].content)

### 🔍 ¿Buscó de verdad?

In [ ]:
ver(response)

## 5 · Seguir la conversación (memoria)
Como estamos en el mismo thread, el chef **recuerda** los ingredientes y las recetas que propuso.

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Dame las instrucciones de la primera receta")]},
    config,
)
print(response["messages"][-1].content)

## 6 · 🎁 Extra: verlo en LangGraph Studio
El mismo chef está en `module-1/1.5_personal_chef.py`, listo para servirse como app. En una terminal, desde la carpeta del repo:

```
cd notebooks/module-1
uv run langgraph dev
```

Se abre **LangGraph Studio** en el navegador: ves el grafo del agente y puedes chatear con él. Para cerrarlo: `Ctrl + C`.

> En el archivo `.py` **no** hay checkpointer: cuando el agente se sirve con `langgraph dev`, la memoria la maneja el servidor.

## ✍️ Tu turno: haz tu propio agente
Usa **M-P-T-M** para crear un agente diferente. Ideas:
- 🎮 Un recomendador de videojuegos que busca reseñas.
- 📚 Un ayudante de tareas que busca fuentes y las cita.
- 🐶 Un experto en mascotas que recuerda el nombre de tu mascota.

Cambia el **system prompt** y la **docstring** de la tool. Pruébalo con dos preguntas seguidas en el mismo thread.

## ✅ Quiz final del módulo 1
1. ¿Cuáles son las 4 piezas de un agente (M-P-T-M)?
2. ¿Quién decide si se usa una tool?
3. ¿Qué pasa si cambias el `thread_id` a mitad de la conversación?
4. ¿Por qué ponemos `max_results=3` en Tavily?
5. ¿Por qué el `.py` servido con `langgraph dev` no lleva checkpointer?

<details><summary>Respuestas</summary>

1. Modelo, Prompt, Tools, Memoria.
2. El modelo, leyendo la docstring.
3. Empieza una conversación nueva: no recuerda nada.
4. Menos texto para leer = respuesta más rápida en CPU.
5. Porque el servidor de LangGraph maneja la persistencia.
</details>

🎉 **¡Terminaste el módulo 1!** Ahora completa las lecciones y quizzes de este módulo en **LangChain Academy**: ahí es donde cuenta para el certificado.